In [1]:
import pickle

In [2]:
with open('/root/autodl-tmp/chuandian_eq/data/taxi/raw/train.pkl', 'rb') as f:
    data = pickle.load(f)

In [3]:
print(data['train'][0])
print(len(data['train']))
print(data['dim_process'])

[{'idx_event': 1, 'type_event': 8, 'time_since_start': 0.0, 'time_since_last_event': 0.0}, {'idx_event': 2, 'type_event': 3, 'time_since_start': 0.07888888888888888, 'time_since_last_event': 0.07888888888888888}, {'idx_event': 3, 'type_event': 8, 'time_since_start': 0.27666666666666667, 'time_since_last_event': 0.19777777777777777}, {'idx_event': 4, 'type_event': 3, 'time_since_start': 0.37972222222222224, 'time_since_last_event': 0.10305555555555557}, {'idx_event': 5, 'type_event': 8, 'time_since_start': 0.5475, 'time_since_last_event': 0.16777777777777775}, {'idx_event': 6, 'type_event': 3, 'time_since_start': 1.0013888888888889, 'time_since_last_event': 0.4538888888888889}, {'idx_event': 7, 'type_event': 8, 'time_since_start': 1.4066666666666667, 'time_since_last_event': 0.40527777777777785}, {'idx_event': 8, 'type_event': 3, 'time_since_start': 1.8002777777777779, 'time_since_last_event': 0.39361111111111113}, {'idx_event': 9, 'type_event': 8, 'time_since_start': 1.8716666666666666

In [4]:
from src.data.sequence import Sequence
import torch
def list_of_dicts_to_sequence(event_list):
    inter_times = [event['time_since_last_event'] for event in event_list]
    inter_times = torch.tensor(inter_times, dtype=torch.float32)
    t_start = 0.0
    t_nll_start = 0.0
    arrival_times = [event['time_since_start'] for event in event_list]
    type_event = [event['type_event'] for event in event_list]
    type_event = torch.tensor(type_event, dtype=torch.long)
    return Sequence(
        t_start=t_start,
        t_nll_start=t_nll_start,
        arrival_times=arrival_times,
        inter_times=inter_times,
        type_event=type_event
    )


In [5]:
sequence_list = [list_of_dicts_to_sequence(s) for s in data["train"]]

/root/autodl-tmp/chuandian_eq/src/data/sequence.py:178: UserWarning: Found 1 zero inter-event times in the sequence. This violates fundamental assumptions of TPP models and may lead to incorrect log-likelihood values.
  warnings.warn(


In [6]:
from src.data.batch import Batch

In [7]:
from src.data.tpp_dataset import TppDataset
ds = TppDataset(sequence_list)
loader = ds.get_dataloader(
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [8]:
for batch in loader:
    print(batch.keys())
    break

['inter_times', 'arrival_times', 't_start', 't_end', 't_nll_start', 'nll_mask', 'start_idx', 'end_idx', 'non_pad_mask', 'type_seq', 'type_event']


In [9]:
batch.type_event[6]

tensor([8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3,
        8, 3, 8, 3, 8, 3, 5, 0, 5, 3, 8, 3, 0, 0])

In [10]:
batch.type_seq

tensor([[   8,    3,    8,  ...,    0, -100, -100],
        [   8,    3,    8,  ...,    3,    8,    1],
        [   5,    3,    8,  ...,    3,    8,    3],
        ...,
        [   8,    3,    8,  ...,    1,    8,    1],
        [   8,    3,    8,  ...,    3,    8,    0],
        [   8,    3,    8,  ...,    1, -100, -100]])

In [11]:
batch.arrival_times

tensor([[0.0000, 0.3361, 0.3739,  ..., 5.3283, 5.3283, 5.3283],
        [0.0000, 0.0986, 0.1550,  ..., 6.7144, 7.1450, 7.2792],
        [0.0000, 0.6828, 0.7444,  ..., 4.9903, 5.0042, 5.1403],
        ...,
        [0.0000, 0.0694, 1.0878,  ..., 7.7081, 7.9878, 8.1983],
        [0.0000, 0.1369, 0.1756,  ..., 8.8917, 9.1222, 9.3786],
        [0.0000, 0.0636, 0.1069,  ..., 7.7378, 7.7378, 7.7378]])

In [12]:
batch.inter_times

tensor([[0.0000, 0.3361, 0.0378,  ..., 0.3306, 0.0000, 0.0000],
        [0.0000, 0.0986, 0.0564,  ..., 0.2517, 0.4306, 0.1342],
        [0.0000, 0.6828, 0.0617,  ..., 0.0869, 0.0139, 0.1361],
        ...,
        [0.0000, 0.0694, 1.0183,  ..., 0.2333, 0.2797, 0.2106],
        [0.0000, 0.1369, 0.0386,  ..., 0.2008, 0.2306, 0.2564],
        [0.0000, 0.0636, 0.0433,  ..., 0.3656, 0.0000, 0.0000]])

In [13]:
from config.config_loader import load_args_from_yaml 
args = load_args_from_yaml("config/THP.yaml")
base_dir = f"data/{args.dataset}"

In [14]:
from src.data.preparation import prepare_data_tpp
df, train_loader, val_loader, test_loader,dataset = prepare_data_tpp(
    args,
    base_dir
)

instantiating registered subclass ChuanDian-SlidingWindow of <class 'src.data.catalog.Catalog'>
Using catalog dataset class: <class 'src.catalogs.chuandian.ChuanDianSlidingWindow'>
Generating the catalog...
Catalog saved to /root/autodl-tmp/chuandian_eq/data/ChuanDian/raw


/root/autodl-tmp/chuandian_eq/src/data/sequence.py:178: UserWarning: Found 1 zero inter-event times in the sequence. This violates fundamental assumptions of TPP models and may lead to incorrect log-likelihood values.
  warnings.warn(


Generated 1858 sliding window sequences.


In [16]:
for batch in test_loader:
    print(f"arival_times: {batch.arrival_times}")
    print(f"inter_times: {batch.inter_times}")
    # print(torch.max(batch.inter_times))

arival_times: tensor([[0.0000, 1.6514, 7.8069,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 5.2550, 8.2816,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 1.8669, 2.9591,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.1240, 3.4458,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 2.9640, 3.0310,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 1.3511, 2.3040,  ..., 0.0000, 0.0000, 0.0000]])
inter_times: tensor([[0.0000, 1.6514, 6.1555,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 5.2550, 3.0265,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 1.8669, 1.0922,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.1240, 3.3218,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 2.9640, 0.0670,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 1.3511, 0.9529,  ..., 0.0000, 0.0000, 0.0000]])
arival_times: tensor([[0.0000, 2.6878, 5.1771,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4530, 0.8788,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.2512, 2.6393,  .